In [1]:
!git clone https://github.com/Break-Through-Tech/MathWorks-1C-unlocking-embedded-ai-through-efficient-compression.git


Cloning into 'MathWorks-1C-unlocking-embedded-ai-through-efficient-compression'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 44 (delta 14), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 19.20 MiB | 25.01 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
!ls MathWorks-1C-unlocking-embedded-ai-through-efficient-compression
!find MathWorks-1C-unlocking-embedded-ai-through-efficient-compression -name "*.mat"

Challenge-Project-Overview.md	notebooks	  src
data				README.md
Getting-Started-for-Fellows.md	requirements.txt
MathWorks-1C-unlocking-embedded-ai-through-efficient-compression/data/test.mat
MathWorks-1C-unlocking-embedded-ai-through-efficient-compression/data/train.mat
MathWorks-1C-unlocking-embedded-ai-through-efficient-compression/data/val.mat


In [3]:
!ls -la MathWorks-1C-unlocking-embedded-ai-through-efficient-compression/notebooks


total 20
drwxr-xr-x 2 root root 4096 Sep 16 22:23 .
drwxr-xr-x 6 root root 4096 Sep 16 22:23 ..
-rw-r--r-- 1 root root 6911 Sep 16 22:23 00_environment_check.ipynb
-rw-r--r-- 1 root root    1 Sep 16 22:23 .gitkeep


In [10]:
!ls data

README.md  test.mat  train.mat	val.mat


In [11]:
# ── Environment Check ──────────────────────────────────────
# Confirms everyone on the team loads the same data, sees the
# same shapes/counts, before splitting into individual work.

import scipy.io
import numpy as np
from collections import Counter

SEED = 42
np.random.seed(SEED)

base = "data"  # adjust path if running outside the repo root

train = scipy.io.loadmat(f"{base}/train.mat")
val   = scipy.io.loadmat(f"{base}/val.mat")
test  = scipy.io.loadmat(f"{base}/test.mat")

X_train = train['trainData']
y_train = [label[0] for label in train['trainLabels'].flatten()]

X_val = val['valData']
y_val = [label[0] for label in val['valLabels'].flatten()]

X_test = test['testData']
y_test = [label[0] for label in test['testLabels'].flatten()]

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Counts:", len(y_train), len(y_val), len(y_test))
print("Train class balance:", Counter(y_train))
print("Val class balance:", Counter(y_val))
print("Test class balance:", Counter(y_test))

assert X_train.shape == (393, 5000)
assert X_val.shape == (27, 5000)
assert X_test.shape == (102, 5000)
assert Counter(y_train) == Counter({'Normal': 131, 'InnerRaceFault': 131, 'OuterRaceFault': 131})
print("\n✅ Data matches spec — environment check passed.")


Shapes: (393, 5000) (27, 5000) (102, 5000)
Counts: 393 27 102
Train class balance: Counter({np.str_('InnerRaceFault'): 131, np.str_('OuterRaceFault'): 131, np.str_('Normal'): 131})
Val class balance: Counter({np.str_('InnerRaceFault'): 9, np.str_('OuterRaceFault'): 9, np.str_('Normal'): 9})
Test class balance: Counter({np.str_('InnerRaceFault'): 34, np.str_('OuterRaceFault'): 34, np.str_('Normal'): 34})

✅ Data matches spec — environment check passed.


In [12]:
import scipy.io
import numpy as np
from collections import Counter
import hashlib

base = "data"

train = scipy.io.loadmat(f"{base}/train.mat")
val   = scipy.io.loadmat(f"{base}/val.mat")
test  = scipy.io.loadmat(f"{base}/test.mat")

X_train = train['trainData']
y_train = [label[0] for label in train['trainLabels'].flatten()]
X_val = val['valData']
y_val = [label[0] for label in val['valLabels'].flatten()]
X_test = test['testData']
y_test = [label[0] for label in test['testLabels'].flatten()]

print("="*70)
print("1. SHAPE / COUNT CHECK")
print("="*70)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Counts:", len(y_train), len(y_val), len(y_test))

expected = {
    'train': ((393, 5000), {'Normal': 131, 'InnerRaceFault': 131, 'OuterRaceFault': 131}),
    'val':   ((27, 5000),  {'Normal': 9, 'InnerRaceFault': 9, 'OuterRaceFault': 9}),
    'test':  ((102, 5000), {'Normal': 34, 'InnerRaceFault': 34, 'OuterRaceFault': 34}),
}
results = {}
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    exp_shape, exp_counts = expected[name]
    shape_ok = (X.shape == exp_shape)
    counts_ok = (dict(Counter(y)) == exp_counts)
    results[name] = shape_ok and counts_ok
    status = "OK" if (shape_ok and counts_ok) else "MISMATCH"
    print(f"  [{name}] expected {exp_shape} {exp_counts} -> {status}")

print()
print("="*70)
print("2. DTYPE / NaN / INF CHECK")
print("="*70)
for name, X in [('train', X_train), ('val', X_val), ('test', X_test)]:
    print(f"[{name}] dtype={X.dtype}, NaN={np.isnan(X).sum()}, Inf={np.isinf(X).sum()}, "
          f"min={X.min():.4f}, max={X.max():.4f}, mean={X.mean():.4f}, std={X.std():.4f}")

print()
print("="*70)
print("3. DUPLICATE CHECK")
print("="*70)
def hash_rows(X):
    return [hashlib.md5(row.tobytes()).hexdigest() for row in X]

h_train, h_val, h_test = hash_rows(X_train), hash_rows(X_val), hash_rows(X_test)

def check_within(name, hashes):
    c = Counter(hashes)
    dupes = {h: cnt for h, cnt in c.items() if cnt > 1}
    print(f"[{name}] within-split duplicates: {len(dupes)}")

check_within('train', h_train)
check_within('val', h_val)
check_within('test', h_test)

set_train, set_val, set_test = set(h_train), set(h_val), set(h_test)
print(f"Cross-split train∩val: {len(set_train & set_val)}")
print(f"Cross-split train∩test: {len(set_train & set_test)}")
print(f"Cross-split val∩test: {len(set_val & set_test)}")

print()
print("ALL SHAPE/COUNT CHECKS PASSED:", all(results.values()))

1. SHAPE / COUNT CHECK
Shapes: (393, 5000) (27, 5000) (102, 5000)
Counts: 393 27 102
  [train] expected (393, 5000) {'Normal': 131, 'InnerRaceFault': 131, 'OuterRaceFault': 131} -> OK
  [val] expected (27, 5000) {'Normal': 9, 'InnerRaceFault': 9, 'OuterRaceFault': 9} -> OK
  [test] expected (102, 5000) {'Normal': 34, 'InnerRaceFault': 34, 'OuterRaceFault': 34} -> OK

2. DTYPE / NaN / INF CHECK
[train] dtype=float64, NaN=0, Inf=0, min=-45.9739, max=39.3167, mean=-0.1538, std=1.2871
[val] dtype=float64, NaN=0, Inf=0, min=-23.5458, max=23.7260, mean=-0.1588, std=1.2301
[test] dtype=float64, NaN=0, Inf=0, min=-26.8856, max=27.4982, mean=-0.1513, std=1.2920

3. DUPLICATE CHECK
[train] within-split duplicates: 0
[val] within-split duplicates: 0
[test] within-split duplicates: 0
Cross-split train∩val: 0
Cross-split train∩test: 0
Cross-split val∩test: 0

ALL SHAPE/COUNT CHECKS PASSED: True


In [13]:
!mkdir -p docs

In [ ]:
%%writefile docs/data-card.md 
# Data Card: Rolling Element Bearing Fault Diagnosis Dataset | Ex:

## Source
- **Data owner:** Eric Bechhoefer (eric@gpms-vt.com)
- **Original source:** data-acoustics.com/measurements/bearing-faults/bearing-2

## License
Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)

## Modifications from Original Dataset
The original dataset contains 6 folders: baseline conditions, outer race faults, inner race faults, analyses, and real-world examples. For this project:
- Only the first 4 folders are used (baseline, outer race, inner race fault conditions) — analyses and real-world examples were dropped.
- Signals resampled to 48,828 Hz.
- Segmented into fixed-length windows of 5,000 samples.
- Balanced across the 3 classes.
- Split into train (75%), val (5%), test (20%).
- Standardization has not been applied yet.

## Known Limitation
Data comes from a single test rig under controlled laboratory conditions.

## Audit Results
- Counts confirmed: train 393 (131/131/131), val 27 (9/9/9), test 102 (34/34/34) — all match spec.
- No NaN or Inf values in any split.
- No duplicate signals within or across splits.

## Open Questions for Advisor
1. Was windowing done before or after the train/val/test split (leakage risk)?
2. What was in the dropped "analyses" and "real-world examples" folders?
3. What resampling method was used to reach 48,828 Hz?

Writing docs/data-card.md


In [ ]:
'''
!git add docs/data-card.md
!git config --global user.email "leowuniv@gmail.com "
!git config --global user.name "leowuniv"
!git commit -m "Add data audit and data card"
!git push
'''
# fix error

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address
